# Hướng dẫn chạy DuplexChat trên Kaggle
Notebook này thiết lập môi trường hoàn chỉnh để chạy pipeline tách nguồn âm thanh (Source Separation) trực tiếp trên nền tảng Kaggle.

### 1. Chuẩn bị môi trường (Cài đặt ffmpeg & uv)

In [1]:
!apt-get update && apt-get install -y ffmpeg
!pip install uv

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://cli.github.com/packages stable InRelease [4,685 B]               
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [112 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,909 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64

### 2. Tải mã nguồn (Git Clone)
Thay thế link GitHub dưới đây bằng link repository mà bạn đã fork nhé!

In [2]:
# Thay bằng link repository của bạn
!git clone https://github.com/ngocbao220/duplex_chat.git
%cd duplex_chat

Cloning into 'duplex_chat'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 103 (delta 19), reused 103 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 219.88 KiB | 2.18 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/kaggle/working/duplex_chat


### 3. Cài đặt các thư viện (Dependencies)
Đồng bộ toàn bộ các packages thông qua `uv`.

In [3]:
!uv sync

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 127 packages in 1ms
Prepared 122 packages in 53.10s                                          
░░░░░░░░░░░░░░░░░░░░ [0/122] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 122 packages in 6.89s                             
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.1
 + aiosignal==1.4.0
 + alembic==1.18.5
 + antlr4-python3-runtime==4.9.3
 + anyio==4.14.1
 + asteroid-filterbanks==0.4.0
 + attrs==26.1.0
 + braceexpand==0.1.7
 + certifi==2026.6.17
 + charset-normalizer==3.4.7
 + click==8.4.2
 + colorlog==6.10.1
 + contourpy==1.3.3
 + cuda-bindings==13.3.1

### 4. Xác thực HuggingFace Token
Bạn cần thêm `HF_TOKEN` vào **Kaggle Secrets** (Menu Add-ons -> Secrets) để quyền tải 2 model Diarization và Separation.

In [4]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    !hf auth login --token $HUGGING_FACE_HUB_TOKEN
except Exception as e:
    print("Lỗi: Không tìm thấy HF_TOKEN trong Kaggle Secrets. Vui lòng vào Add-ons -> Secrets để thêm bí mật có tên 'HF_TOKEN'.")


  A new version of huggingface_hub is available: 1.11.0 → 1.29.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) ^C

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `write_to_object_detection` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 5. Chạy end-to-end test theo model
Chọn model trong `ENABLED_TEST_NAMES`; mỗi lần chạy lưu audio vào thư mục riêng theo đúng model để nghe lại.

In [8]:
!uv add wrapt

Resolved 128 packages in 1.43s                                       
Prepared 2 packages in 724ms                                             ⠋ Preparing packages... (0/0)                                                   
Uninstalled 1 package in 0.56ms
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 4ms                                 
 ~ duplexchat-pipe==0.1.0 (from file:///kaggle/working/duplex_chat)
 + wrapt==2.3.0


In [9]:
!uv run python -c "import wrapt, numpy; print('wrapt OK'); print(numpy.__version__)"

wrapt OK
2.5.1


In [41]:
!git pull

Already up to date.


In [ ]:
from pathlib import Path

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_TESTS = [
    {
        "name": "pyannote31__dialoguesidon",
        "install": "uv sync --extra diarization-pyannote --extra separation-dialoguesidon",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "sortformer__dialoguesidon",
        "install": "uv sync --extra diarization-sortformer --extra separation-dialoguesidon",
        "diarization_backend": "sortformer",
        "diarization_model": "nvidia/diar_sortformer_4spk-v1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "diarizen__dialoguesidon",
        "install": "uv pip install -r requirements/diarization-diarizen.txt && uv sync --extra separation-dialoguesidon",
        "diarization_backend": "diarizen",
        "diarization_model": "BUT-FIT/diarizen-wavlm-large-s80-md",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "pyannote31__sepformer",
        "install": "uv sync --extra diarization-pyannote --extra separation-sepformer",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "sepformer",
        "separation_model": "speechbrain/sepformer-wsj02mix",
    },
    {
        "name": "pyannote31__mossformer2",
        "install": "uv pip install -r requirements/separation-mossformer2.txt && uv sync --extra diarization-pyannote",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "mossformer2",
        "separation_model": "alibabasglab/MossFormer2_SS_16K",
    },
]

# Chạy tất cả nếu muốn: ENABLED_TEST_NAMES = [test["name"] for test in MODEL_TESTS]
ENABLED_TEST_NAMES = ["pyannote31__dialoguesidon"]

for test in MODEL_TESTS:
    if test["name"] not in ENABLED_TEST_NAMES:
        continue
    out_dir = OUT_ROOT / test["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    output_prefix = out_dir / "speaker"
    print(f"\n=== Running {test['name']} ===")
    print(f"Audio will be saved under: {out_dir}")
    install_cmd = test.get("install")
    if install_cmd:
        !{install_cmd}
    !MPLBACKEND=Agg uv run python test_single.py "$AUDIO_PATH"         --diarize-chunk 240         --separate-chunk 240         --diarization-backend "{test['diarization_backend']}"         --diarization-model "{test['diarization_model']}"         --separation-backend "{test['separation_backend']}"         --separation-model "{test['separation_model']}"         --output-prefix "$output_prefix"


### 6. Nghe lại kết quả

In [ ]:
from pathlib import Path
import IPython.display as ipd

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")

print("Audio gốc:")
display(ipd.Audio(AUDIO_PATH))

for out_dir in sorted(OUT_ROOT.glob("*")):
    spk_a = out_dir / "speaker_A.wav"
    spk_b = out_dir / "speaker_B.wav"
    if not spk_a.exists() or not spk_b.exists():
        continue
    print(f"\nModel test: {out_dir.name}")
    print("Giọng Người A:")
    display(ipd.Audio(str(spk_a)))
    print("Giọng Người B:")
    display(ipd.Audio(str(spk_b)))
